# `.to()` in PyTorch — quick summary

## What it controls

`.to()` changes two things about a tensor (or model):

1. **Device** — where it lives (CPU or GPU)
2. **Dtype** — what data type it holds (float32, int64, etc.)

```python
x = x.to("cuda")                          # change device
x = x.to(torch.float32)                   # change dtype
x = x.to(device="cuda", dtype=torch.float32)  # both at once
```

---

## Tensors: NOT in-place — you must reassign

```python
x = torch.rand(3)

x.to("cuda")        # ❌ result discarded, x is unchanged
x = x.to("cuda")    # ✅ correct, reassign the result
```

A plain tensor's `.to()` always returns a **new** tensor. The original is untouched unless you reassign it.

---

## Models (`nn.Module`): IS in-place — modifies all parameters directly

```python
model = MyModel()          # all weights start on CPU
model.to("cuda")           # this alone already moves everything, in-place
```

When called on a model, `.to()` recursively walks through **every parameter and buffer** in the model — every layer, every sub-module — and moves/converts each one in place. It's still common (and safe) to write `model = model.to(device)`, but the reassignment isn't strictly required.

```python
model = model.to(device)                   # move all weights to a device
model = model.to(torch.float16)            # convert all weights to float16
model = model.to(device=device, dtype=torch.float16)  # both
```

Checking a model's device (there's no single `.device` attribute on a model):

```python
next(model.parameters()).device
```

---

## Key rule: model and data must be on the same device

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)          # move the model ONCE, before the training loop

for x_batch, y_batch in loader:
    x_batch = x_batch.to(device)  # move data EVERY batch, inside the loop
    y_batch = y_batch.to(device)

    pred = model(x_batch)
```

Forgetting to move one of them raises:
```
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
```

---

## Summary table

| | Tensor | Model (`nn.Module`) |
|---|---|---|
| In-place? | No — returns a new tensor | Yes — modifies parameters directly |
| Must reassign? | Yes, required | No, but common practice |
| What gets moved | The tensor itself | All parameters/buffers, recursively |
| When to call it | Every batch | Once, before the training loop |

In [1]:
import torch

a = torch.tensor(2)

print(f'Before the adjusts: {a.dtype}, {a.device}')

Before the adjusts: torch.int64, cpu


In [ ]:
device = 'cuda'
dtype = torch.float32

a = a.to(device, dtype)

print(f'After the adjusts: {a.dtype}, {a.device}')

After the adjusts: torch.float32, cuda:0


: 